# 真实数据案例：网络构建与统计验证

来自《生物医学网络科学》（Biomedical Network Science）。数据文件放在与本笔记本同级的 `data/` 文件夹中。

# 从表达矩阵到共表达网络：相关、偏相关、阈值选择与统计验证

本案例从 1206 个人类细胞系的 RNA 测序数据出发，构建 56 个基因的共表达网络，并逐步检验每一个构网决定：用相关还是偏相关、阈值怎么定、多重比较怎么校正、结果在重抽样下是否稳定、样本量小时会怎样。所用的公式（Fisher $z$ 变换、BH 程序、偏相关、经验 $p$ 值、bootstrap）与[网络构建与统计验证](network-construction.html)专题页一致。

**1. 数据来源**

- 数据：Human Protein Atlas（HPA，proteinatlas.org）细胞系转录组数据 `rna_celline.tsv`，下载地址 `https://www.proteinatlas.org/download/tsv/rna_celline.tsv.zip`（2026-11-20 下载，压缩包内文件日期为 2023-05-24）。这是 HPA 对 1206 个细胞系做的 RNA-seq，表达量为标准化后的 nTPM。
- 许可证：HPA 声明其数据库中可受版权保护的部分按 **CC BY 4.0** 许可（`https://www.proteinatlas.org/about/licence`）。引用：Uhlén M 等, Tissue-based map of the human proteome, *Science* 2015, DOI [10.1126/science.1260419](https://doi.org/10.1126/science.1260419)；Jin H 等, Systematic transcriptional analysis of human cell lines for gene expression landscape and tumor representation, *Nature Communications* 2023, DOI [10.1038/s41467-023-41132-w](https://doi.org/10.1038/s41467-023-41132-w)。
- 裁剪：只保留 56 个基因的 nTPM，整理成“细胞系 × 基因”的宽表 `data/hpa_celline_rna_panel56.csv`（保留一位小数）。
- 基因的选择：按常用的标记基因归为 7 组，每组 8 个——上皮（EPCAM、CDH1、角蛋白、紧密连接蛋白等）、间质（VIM、ZEB1、胶原等）、造血细胞（PTPRC 等）、MHC II 类抗原提呈（HLA-DR/DP/DM、CD74、CIITA）、细胞增殖（MKI67、TOP2A、周期蛋白等）、神经/神经内分泌（CHGA、SYP、ASCL1 等）、干扰素刺激基因（IFIT1、ISG15、MX1 等）。这一分组是**分析前定下的先验标签**，只用来检验构出的网络能否找回已知的共表达模块，不是网络的“真值”。

## 2. 读入与清洗

表达量是右偏的计数型数据，先做 $\log_2(\mathrm{nTPM}+1)$ 变换。再检查每个基因在多少细胞系里几乎不表达（nTPM < 1）：大量零值会让皮尔逊相关被少数高表达样本主导，这是解读时要记住的一点。

In [ ]:
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy import stats

t0 = time.time()
X = pd.read_csv("data/hpa_celline_rna_panel56.csv", index_col=0)
print("细胞系 × 基因:", X.shape, "；缺失值:", int(X.isna().sum().sum()))
Y = np.log2(X + 1.0)
genes = list(Y.columns)
p, n = len(genes), len(Y)
group_names = ["epithelial", "mesenchymal", "hematopoietic", "MHC-II", "proliferation", "neuro/NE", "interferon"]
grp = np.repeat(np.arange(7), 8)                     # 文件中基因已按组排列，每组 8 个
low = (X < 1).mean()
print("nTPM < 1 的细胞系比例：中位数 %.2f，最高 %.2f（%s）" % (low.median(), low.max(), low.idxmax()))
for g in range(7):
    print("  %-14s %s" % (group_names[g], ", ".join(genes[8 * g:8 * g + 8])))

## 3. 构网：相关矩阵与偏相关矩阵

- **皮尔逊相关** $r_{ij}$：间接关联也会成边（$A\to B\to C$ 时 $A$、$C$ 也相关）。
- **偏相关** $\rho_{ij\cdot\text{rest}}=-\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$，$\Theta=\Sigma^{-1}$ 为精度矩阵。这里 $n=1206$ 远大于 $p=56$，样本协方差矩阵可以直接求逆；若 $n$ 与 $p$ 同量级，就需要收缩估计或图形 LASSO（见[连接强度的估计](network-construction.edge-estimation.html)）。
- 作为稳健性检查，再算一次斯皮尔曼秩相关，看它与皮尔逊相关是否一致。

In [ ]:
Yv = Y.values
R = np.corrcoef(Yv, rowvar=False)
Theta = np.linalg.inv(np.cov(Yv, rowvar=False))
dg = np.sqrt(np.diag(Theta))
P = -Theta / np.outer(dg, dg)
np.fill_diagonal(P, 1.0)
iu = np.triu_indices(p, 1)
r, pr = R[iu], P[iu]
rs = stats.spearmanr(Yv).correlation[iu]
same = grp[iu[0]] == grp[iu[1]]
print("基因对数 m = p(p-1)/2 = %d，其中同组 %d 对" % (len(r), same.sum()))
print("同组 |r| 中位数 %.2f，异组 |r| 中位数 %.2f" % (np.median(abs(r[same])), np.median(abs(r[~same]))))
print("皮尔逊与斯皮尔曼相关的相关: %.3f" % np.corrcoef(r, rs)[0, 1])
print("|r| > 0.5 的对数 %d；|偏相关| > 0.2 的对数 %d" % ((abs(r) > 0.5).sum(), (abs(pr) > 0.2).sum()))

## 4. 核心步骤：逐边检验、多重比较与阈值选择

**单条边的检验**。在零假设 $\rho=0$ 下，Fisher 变换 $z=\operatorname{artanh}(r)$ 近似正态，方差约 $1/(n-3)$；对偏相关，要再减去被控制的变量数 $p-2$，方差约 $1/(n-(p-2)-3)$。

**多重比较**。一共 1540 次检验，用 BH 程序把 FDR 控制在 $q=0.05$；另用置换的最大统计量法控制族错误率（FWE）：每次把每个基因的取值在细胞系之间独立打乱（破坏所有基因间的依赖），记录所有基因对中最大的 $|r|$，取 200 次置换的 95% 分位数作为阈值。

然后把几种常见的构网规则放在一起比较：显著性阈值（BH、FWE）、绝对阈值 $|r|>0.5$、偏相关显著性，以及保留最强 10% 的比例阈值。每种规则下计算边数、密度、正边比例、同组边比例和平均聚集系数。

In [ ]:
def bh(pvals, q=0.05):
    m = len(pvals)
    o = np.argsort(pvals)
    ok = pvals[o] <= q * np.arange(1, m + 1) / m
    k = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
    rej = np.zeros(m, bool)
    rej[o[:k]] = True
    return rej

p_r = 2 * stats.norm.sf(np.abs(np.arctanh(r)) * np.sqrt(n - 3))
p_pr = 2 * stats.norm.sf(np.abs(np.arctanh(pr)) * np.sqrt(n - (p - 2) - 3))

rng = np.random.default_rng(2026)
B_perm = 200
max_abs = np.empty(B_perm)
for b in range(B_perm):
    Yp = np.column_stack([rng.permutation(Yv[:, j]) for j in range(p)])
    max_abs[b] = np.abs(np.corrcoef(Yp, rowvar=False)[iu]).max()
tau_fwe = np.quantile(max_abs, 0.95)
print("置换零分布：最大 |r| 的 95%% 分位数 = %.3f" % tau_fwe)

def graph_from(mask, w):
    G = nx.Graph()
    G.add_nodes_from(range(p))
    G.add_weighted_edges_from((i, j, x) for i, j, x in zip(iu[0][mask], iu[1][mask], w[mask]))
    return G

top10 = np.zeros(len(r), bool)
top10[np.argsort(-np.abs(r))[: int(round(0.10 * len(r)))]] = True
rules = {
    "corr, BH q<0.05": (bh(p_r), r),
    "corr, perm FWE": (np.abs(r) > tau_fwe, r),
    "corr, |r|>0.5": (np.abs(r) > 0.5, r),
    "corr, top 10%": (top10, r),
    "partial, BH q<0.05": (bh(p_pr), pr),
}
rows = []
for name, (mask, w) in rules.items():
    G = graph_from(mask, w)
    rows.append(dict(rule=name, edges=int(mask.sum()), density=nx.density(G), pos_frac=(w[mask] > 0).mean(),
                     within_group=same[mask].mean(), clustering=nx.average_clustering(G)))
table = pd.DataFrame(rows).set_index("rule")
print(table.round(3).to_string())
print("说明：随机连边时同组边比例的期望 = %.3f" % same.mean())

**社团能否找回先验分组？** 在 $|r|>0.5$ 的网络（只用正边）和偏相关 BH 网络（只用正边）上用 Louvain 算法找社团，用调整兰德指数（ARI，1 为完全一致、0 为随机水平）与 7 个先验组比较；同时计算先验分组本身的模块度 $Q$。

In [ ]:
def ari(a, b):
    a, b = np.asarray(a), np.asarray(b)
    _, ai = np.unique(a, return_inverse=True)
    _, bi = np.unique(b, return_inverse=True)
    C = np.zeros((ai.max() + 1, bi.max() + 1))
    np.add.at(C, (ai, bi), 1)
    c2 = lambda x: x * (x - 1) / 2
    s, sa, sb = c2(C).sum(), c2(C.sum(1)).sum(), c2(C.sum(0)).sum()
    e = sa * sb / c2(len(a))
    return (s - e) / (0.5 * (sa + sb) - e)

prior = [set(np.where(grp == g)[0]) for g in range(7)]
for name in ["corr, |r|>0.5", "partial, BH q<0.05"]:
    mask, w = rules[name]
    G = graph_from(mask & (w > 0), w)
    comms = nx.community.louvain_communities(G, seed=1)
    lab = np.empty(p, int)
    for c, S in enumerate(comms):
        lab[list(S)] = c
    print("%-20s 正边 %3d 条：Louvain 社团 %d 个，Q = %.3f；先验分组的 Q = %.3f；ARI = %.2f"
          % (name, G.number_of_edges(), len(comms), nx.community.modularity(G, comms),
             nx.community.modularity(G, prior), ari(lab, grp)))
G_main = graph_from(rules["corr, |r|>0.5"][0] & (r > 0), r)

## 5. 与零模型比较

两个问题、两个零模型：

1. **“这些边是不是噪声？”**——用上一步的置换零模型：在打乱后的数据上用同样的 BH 规则构网，期望几乎没有边。
2. **“这个网络的聚集是否超出度序列能解释的程度？”**——对 $|r|>0.5$ 的正边网络做保度随机化（双边交换，每次交换 $10L$ 次，100 个随机网络），比较平均聚集系数和先验分组的模块度，报告 $z$ 分数与经验 $p$ 值 $\hat p=(1+\#\{X_m\ge X_{\rm obs}\})/(M+1)$。

要注意专题页强调的一点：相关矩阵必须半正定，强相关具有“传递性”，所以相关网络的聚集系数本来就会高于保度随机网络（Zalesky 等 2012）。保度零模型下显著，不能直接说明有特殊的生物学组织；先验分组的模块度高，才是“找回了已知模块”的证据。

In [ ]:
n_null_edges = []
for b in range(20):
    Yp = np.column_stack([rng.permutation(Yv[:, j]) for j in range(p)])
    rp = np.corrcoef(Yp, rowvar=False)[iu]
    n_null_edges.append(bh(2 * stats.norm.sf(np.abs(np.arctanh(rp)) * np.sqrt(n - 3))).sum())
print("置换数据上 BH 网络的边数（20 次）: 均值 %.2f，最大 %d" % (np.mean(n_null_edges), max(n_null_edges)))

M_null = 100
C_obs = nx.average_clustering(G_main)
Q_obs = nx.community.modularity(G_main, prior)
Lm = G_main.number_of_edges()
C_null, Q_null = [], []
for b in range(M_null):
    H = G_main.copy()
    nx.double_edge_swap(H, nswap=10 * Lm, max_tries=100 * Lm, seed=int(rng.integers(1 << 31)))
    C_null.append(nx.average_clustering(H))
    Q_null.append(nx.community.modularity(H, prior))
C_null, Q_null = np.array(C_null), np.array(Q_null)
for name, x, xn in [("平均聚集系数", C_obs, C_null), ("先验分组模块度", Q_obs, Q_null)]:
    z = (x - xn.mean()) / xn.std(ddof=1)
    ph = (1 + (xn >= x).sum()) / (M_null + 1)
    print("%-8s 观测 %.3f；保度零模型 %.3f ± %.3f；z = %.1f；经验 p = %.3f" % (name, x, xn.mean(), xn.std(ddof=1), z, ph))

**稳定性：bootstrap 与样本量。** 有放回地重抽样细胞系 200 次，把“相关 → 阈值”整个流程放进循环，记录每条边的入选频率。入选频率在 0.2–0.8 之间的边，就是“阈值附近、入选与否近似抛硬币”的边。再随机抽 50 个细胞系重复构网，看小样本下 BH 网络与全样本网络的差别——这对应专题页“样本量与统计功效”的讨论。

In [ ]:
B_boot = 200
freq_abs = np.zeros(len(r))
freq_pbh = np.zeros(len(r))
C_boot = []
for b in range(B_boot):
    ix = rng.integers(0, n, n)
    Yb = Yv[ix]
    rb = np.corrcoef(Yb, rowvar=False)[iu]
    Tb = np.linalg.inv(np.cov(Yb, rowvar=False))
    db = np.sqrt(np.diag(Tb))
    prb = (-Tb / np.outer(db, db))[iu]
    mb = np.abs(rb) > 0.5
    freq_abs += mb
    freq_pbh += bh(2 * stats.norm.sf(np.abs(np.arctanh(prb)) * np.sqrt(n - (p - 2) - 3)))
    C_boot.append(nx.average_clustering(graph_from(mb & (rb > 0), rb)))
freq_abs /= B_boot
freq_pbh /= B_boot
for name, f, mask in [("|r|>0.5", freq_abs, rules["corr, |r|>0.5"][0]), ("偏相关 BH", freq_pbh, rules["partial, BH q<0.05"][0])]:
    print("%-10s 全样本选中 %d 条；bootstrap 入选频率 >= 0.95 的边 %d 条；0.2–0.8 之间 %d 条"
          % (name, mask.sum(), (f >= 0.95).sum(), ((f > 0.2) & (f < 0.8)).sum()))
print("|r|>0.5 正边网络的平均聚集系数：全样本 %.3f，bootstrap 95%% 区间 [%.3f, %.3f]"
      % (C_obs, *np.quantile(C_boot, [0.025, 0.975])))

full_bh = bh(p_r)
for n_sub in [50, 200]:
    found, overlap = [], []
    for b in range(20):
        ix = rng.choice(n, n_sub, replace=False)
        rsub = np.corrcoef(Yv[ix], rowvar=False)[iu]
        sel = bh(2 * stats.norm.sf(np.abs(np.arctanh(rsub)) * np.sqrt(n_sub - 3)))
        found.append(sel.sum())
        overlap.append((sel & full_bh).sum() / max(sel.sum(), 1))
    print("n = %4d：BH 网络平均 %.0f 条边（全样本 %d 条），其中 %.1f%% 也在全样本网络里"
          % (n_sub, np.mean(found), full_bh.sum(), 100 * np.mean(overlap)))

## 6. 作图

上左：按先验分组排列的相关矩阵，组内方块清晰可见。上右：$|r|>0.5$ 正边网络，颜色为先验分组。下左：在 5%–30% 的密度范围内扫描比例阈值，看平均聚集系数和先验分组模块度如何随密度变化（每个密度下也算一次保度零模型的均值作对照）。下右：两种规则下 bootstrap 入选频率的分布。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax = axes[0, 0]
im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1)
for k in range(1, 7):
    ax.axhline(8 * k - 0.5, color="k", lw=0.6)
    ax.axvline(8 * k - 0.5, color="k", lw=0.6)
ax.set_xticks(np.arange(4, 56, 8)); ax.set_xticklabels(group_names, rotation=45, ha="right", fontsize=8)
ax.set_yticks(np.arange(4, 56, 8)); ax.set_yticklabels(group_names, fontsize=8)
ax.set_title("Pearson correlation, log2(nTPM+1), n = %d cell lines" % n)
plt.colorbar(im, ax=ax, fraction=0.046)

ax = axes[0, 1]
cols = plt.cm.tab10(np.arange(7))
pos = nx.spring_layout(G_main, seed=3, k=0.35)
nx.draw_networkx_edges(G_main, pos, ax=ax, alpha=0.35)
nx.draw_networkx_nodes(G_main, pos, ax=ax, node_size=90, node_color=[cols[grp[i]] for i in G_main])
nx.draw_networkx_labels(G_main, pos, ax=ax, labels={i: genes[i] for i in G_main}, font_size=6)
for g in range(7):
    ax.scatter([], [], color=cols[g], label=group_names[g])
ax.legend(fontsize=7, loc="lower left"); ax.axis("off")
ax.set_title("|r| > 0.5, positive edges (%d)" % G_main.number_of_edges())

ax = axes[1, 0]
dens = np.arange(0.05, 0.31, 0.025)
order = np.argsort(-r)                     # 只取正相关，从强到弱
Cs, Qs, Cn, Qn = [], [], [], []
for d in dens:
    k = int(round(d * len(r)))
    mask = np.zeros(len(r), bool); mask[order[:k]] = True
    G = graph_from(mask, r)
    Cs.append(nx.average_clustering(G)); Qs.append(nx.community.modularity(G, prior))
    cn, qn = [], []
    for b in range(10):
        H = G.copy()
        nx.double_edge_swap(H, nswap=5 * k, max_tries=100 * k, seed=int(rng.integers(1 << 31)))
        cn.append(nx.average_clustering(H)); qn.append(nx.community.modularity(H, prior))
    Cn.append(np.mean(cn)); Qn.append(np.mean(qn))
ax.plot(dens, Cs, "o-", label="clustering (data)")
ax.plot(dens, Cn, "o--", label="clustering (degree-preserving)")
ax.plot(dens, Qs, "s-", label="Q of prior groups (data)")
ax.plot(dens, Qn, "s--", label="Q of prior groups (degree-preserving)")
ax.set_xlabel("edge density (strongest positive correlations kept)"); ax.legend(fontsize=8)
ax.set_title("Density sweep 5%-30%")

ax = axes[1, 1]
for f, lab in [(freq_abs, "|r| > 0.5"), (freq_pbh, "partial corr, BH")]:
    ax.hist(f[f > 0], bins=20, range=(0, 1), alpha=0.6, label=lab)
ax.set_xlabel("bootstrap selection frequency (edges selected at least once)"); ax.set_ylabel("edges")
ax.set_yscale("log"); ax.legend(); ax.set_title("Edge stability, %d bootstrap resamples" % B_boot)
plt.tight_layout()
print("总运行时间 %.1f 秒" % (time.time() - t0))

## 7. 解读与局限

**结果怎么读**（具体数字以上面的运行输出为准）：

- **统计显著不等于强关联。** $n=1206$ 时，BH 校正后约八成的基因对都“显著”，网络几乎是稠密的，而且大部分边在组间、过半是负相关。置换的最大统计量阈值也只有 $|r|\approx0.12$。样本很大时，显著性阈值回答的是“相关是否恰好为 0”，并不能帮助挑出有意义的结构；这时需要同时考虑效应量（如 $|r|>0.5$）或改用偏相关。
- **置换零模型下几乎没有边**，说明 BH 网络里的边不是随机噪声造成的；问题不在假阳性，而在“真实但很弱、多为间接”的关联太多。
- **绝对阈值 $|r|>0.5$ 的网络**以正边、同组边为主，Louvain 社团与先验分组高度一致，先验分组的模块度也远高于保度零模型。这些组大致对应细胞系的来源谱系（上皮、间质、造血、神经内分泌）以及增殖、干扰素反应、抗原提呈等程序。
- **偏相关**控制其余 54 个基因后，许多组间的间接关联消失，但仍有大量弱的偏相关通过 BH 校正（其中不少是负的组间边，所以同组边比例不高）。只看正的偏相关边时，Louvain 社团与 7 个先验组完全一致（ARI = 1）：组内的“直接关联”是正的，组间关联多半经由其他基因传递。专题页也提醒：偏相关仍然不是因果，还可能在对撞结构中制造关联。
- **聚集系数**在保度零模型下显著偏高，但相关网络的传递性本来就会抬高聚集系数，这一结果单独不能作为“特殊组织”的证据。密度扫描显示，聚集系数和模块度都随密度改变，结论应在一段密度范围内报告，而不是只报一个阈值。
- **稳定性**：$|r|>0.5$ 网络的大多数边在 bootstrap 中入选频率接近 1，只有少数阈值附近的边入选频率在 0.2–0.8 之间；偏相关 BH 网络则不同，频率在 0.2–0.8 之间的边比全样本选中的边还多——这些都是很弱的偏相关，是否“显著”几乎取决于抽到哪些细胞系。整体指标可以比单条边稳定得多：$|r|>0.5$ 网络平均聚集系数的 bootstrap 区间很窄。小样本（$n=50$）时 BH 网络只剩下最强的一部分边，而且这些边几乎都出现在全样本网络里；所以“在小样本里没有边”不代表没有关联，只是功效不足。

**局限**：

- 细胞系不是组织：多年体外培养会改变表达，不同来源的细胞系在库中的比例也不均衡。相关性主要由“这是哪一类细胞系”驱动，网络因此更像“谱系标记的共现图”，而不是细胞内的调控网络。若要研究某一种细胞内的共表达，应在同一谱系内部构网，或把谱系作为协变量回归掉。
- 56 个基因是按先验分组挑选的，先验分组能被找回部分是选择方式造成的；在全基因组规模上（$p\gg n$）需要收缩估计、图形 LASSO 或 WGCNA 一类的方法，阈值和多重比较的负担也完全不同。
- $\log_2(x+1)$ 变换后，大量接近 0 的值仍会影响皮尔逊相关；斯皮尔曼相关与之高度一致，说明主要结论不依赖这一点，但没有检验更复杂的非线性依赖（如互信息）。
- Fisher $z$ 检验假设样本独立、近似二元正态；来自同一患者或同一实验室的细胞系并不完全独立，有效样本量可能小于 1206。